# RO3 — Explainable AI + Sentiment-Price Temporal Dynamics

**Research objective (RO3):** *"To establish and evaluate an explainable AI framework capable of delivering
transparent and interpretable predictions in stock selection, analyzing the impact of sentiment data noise
and exploring the temporal dynamics between sentiment shifts and stock price movements."*

**Base papers:**
- *A Controlled Experiment on SHAP-Based Explainable AI for Portfolio Rebalancing* — SHAP explanations over a
  tree-model's predictions, used to make trading decisions more transparent and (in the original human-subject
  study) more trustworthy.
- *A Dynamic-Causal Hybrid Framework for NIFTY50 Stock Decision Making* — its "mechanistically explainable,
  not a black box" angle motivates treating explainability as a first-class evaluation axis here, alongside
  accuracy.

**What this notebook does (simplified, real-data version):**
0. Targets the **21-trading-day ("~30-day swing") forward return**, not next-day — this project's production
   ledger found no usable edge at 1-day horizons and real, gate-able edge at a ~30-day swing horizon instead,
   so this notebook's explainability findings are computed on the horizon where there is something real to
   explain, rather than mostly noise.
1. Trains a **LightGBM** regressor on real technical + fundamental + macro features (full 2015-today history,
   large honest sample) and explains it with **SHAP** (global summary, local waterfalls for real specific
   dates, a dependence plot).
2. Runs a **second, smaller model** on the recent window where real sentiment coverage exists (same
   constraint as Notebooks 1-2), and uses it for two sentiment-specific diagnostics:
   - **Sentiment noise robustness**: inject increasing Gaussian noise into the sentiment feature *at inference
     only* and watch accuracy and SHAP-attributed importance degrade — a real, standard robustness probe.
   - **Temporal dynamics**: Granger-causality test and a cross-correlation function between the sentiment gate
     and forward returns at lags 1-5 days.
3. An illustrative **decision-quality** experiment: skip trades where the top-2 SHAP drivers disagree in sign
   (an algorithmic proxy for "the explanation is incoherent, don't trust it"), and honestly compare the
   resulting backtest against trading on every prediction.

We do not expect the sentiment-window results to be statistically powerful (same free-data constraint as
Notebooks 1-2) — they are reported with sample sizes attached, not polished into a bigger claim than the data
supports.

Runtime: ~2-3 minutes on a free Colab T4 (or CPU-only Colab runtime — LightGBM/SHAP here don't need a GPU).

In [ ]:
!pip -q install yfinance==0.2.* lightgbm shap statsmodels --upgrade
import warnings; warnings.filterwarnings("ignore")
print("done")

In [ ]:
import numpy as np
import pandas as pd
import requests, io
import matplotlib.pyplot as plt
import yfinance as yf
import lightgbm as lgb
import shap
from sklearn.metrics import mean_absolute_error, r2_score
from statsmodels.tsa.stattools import grangercausalitytests

SEED = 7
np.random.seed(SEED)

TICKERS = ["RELIANCE.NS","TCS.NS","HDFCBANK.NS","INFY.NS","ICICIBANK.NS","ITC.NS","LT.NS","SBIN.NS",
           "BHARTIARTL.NS","HINDUNILVR.NS"]
START = "2015-01-01"
HORIZON = 21   # ~30 calendar days -- the swing horizon where this project's production ledger found real edge
TRAIN_END, VAL_END = "2022-01-01", "2023-01-01"

### 1. Real data: prices+technicals, fundamentals, macro (same sources as Notebook 1)

In [ ]:
def fetch_prices(ticker, start=START):
    df = yf.download(ticker, start=start, progress=False, auto_adjust=True)
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    return df.dropna(how="all")

def add_technical(df):
    out = df.copy()
    out["ret1"] = out["Close"].pct_change()
    out["sma20"] = out["Close"].rolling(20).mean() / out["Close"] - 1
    out["ema20"] = out["Close"].ewm(span=20).mean() / out["Close"] - 1
    delta = out["Close"].diff()
    up = delta.clip(lower=0).rolling(14).mean()
    down = (-delta.clip(upper=0)).rolling(14).mean()
    rs = up / down.replace(0, np.nan)
    out["rsi14"] = (100 - (100/(1+rs))) / 100.0
    ema12, ema26 = out["Close"].ewm(span=12).mean(), out["Close"].ewm(span=26).mean()
    out["macd"] = (ema12 - ema26) / out["Close"]
    std20 = out["Close"].rolling(20).std()
    out["bb_pctb"] = (out["Close"] - (out["sma20"]*out["Close"]+out["Close"] - 2*std20)) / (4*std20 + 1e-9)
    out["vol20"] = out["ret1"].rolling(20).std() * np.sqrt(252)
    out["fwd_ret"] = out["Close"].shift(-HORIZON) / out["Close"] - 1
    return out

TECH_COLS = ["ret1","sma20","ema20","rsi14","macd","bb_pctb","vol20"]
price_data = {tk: add_technical(fetch_prices(tk)) for tk in TICKERS}

def fetch_fundamentals(ticker):
    t = yf.Ticker(ticker)
    qf = t.quarterly_financials
    fdf = pd.DataFrame()
    if qf is not None and not qf.empty:
        rows = {k: qf.loc[k].sort_index() for k in ["Total Revenue","Net Income"] if k in qf.index}
        if rows:
            fdf = pd.DataFrame(rows).sort_index()
            fdf["rev_growth_qoq"] = fdf.get("Total Revenue", pd.Series(dtype=float)).pct_change()
            fdf["ni_growth_qoq"] = fdf.get("Net Income", pd.Series(dtype=float)).pct_change()
    try:
        ed = t.get_earnings_dates(limit=40)[["Surprise(%)"]].dropna().sort_index()
        ed.index = ed.index.tz_localize(None)
        ed.columns = ["eps_surprise_pct"]
    except Exception:
        ed = pd.DataFrame(columns=["eps_surprise_pct"])
    return fdf, ed

FUND_COLS = ["rev_growth_qoq","ni_growth_qoq","eps_surprise_pct"]
fund_data, earnings_dates = {}, {}
for tk in TICKERS:
    fdf, ed = fetch_fundamentals(tk)
    idx = price_data[tk].index
    rev = fdf["rev_growth_qoq"].reindex(idx, method="ffill") if "rev_growth_qoq" in fdf else pd.Series(np.nan, index=idx)
    ni  = fdf["ni_growth_qoq"].reindex(idx, method="ffill") if "ni_growth_qoq" in fdf else pd.Series(np.nan, index=idx)
    eps = ed["eps_surprise_pct"].reindex(idx, method="ffill") if not ed.empty else pd.Series(np.nan, index=idx)
    fund_data[tk] = pd.DataFrame({"rev_growth_qoq": rev, "ni_growth_qoq": ni, "eps_surprise_pct": eps})
    earnings_dates[tk] = ed

def fetch_fred(series_id):
    r = requests.get(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}", timeout=20)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text)); df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"]); df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    return df.set_index("date")[series_id].dropna()

usdinr_chg5 = fetch_fred("DEXINUS").pct_change(5)
gdp_growth = fetch_fred("INDGDPRQPSMEI"); gdp_growth.index = gdp_growth.index + pd.Timedelta(days=60)
RBI_REPO_RATE = pd.DataFrame({
    "date": ["2014-01-01","2015-01-15","2015-03-04","2015-06-02","2015-09-29","2016-04-05","2017-08-02",
             "2018-06-06","2018-08-01","2019-02-07","2019-04-04","2019-06-06","2019-08-07","2019-10-04",
             "2020-03-27","2020-05-22","2022-05-04","2022-06-08","2022-08-05","2022-09-30","2022-12-07","2023-02-08"],
    "repo_rate": [8.00,7.75,7.50,7.25,6.75,6.50,6.00,6.25,6.50,6.25,6.00,5.75,5.40,5.15,
                  4.40,4.00,4.40,4.90,5.40,5.90,6.25,6.50],
})
RBI_REPO_RATE["date"] = pd.to_datetime(RBI_REPO_RATE["date"])
repo_series = RBI_REPO_RATE.set_index("date")["repo_rate"].sort_index()
MACRO_COLS = ["usdinr_chg5","gdp_growth","repo_rate"]
macro_data = {}
for tk in TICKERS:
    idx = price_data[tk].index
    macro_data[tk] = pd.DataFrame({"usdinr_chg5": usdinr_chg5.reindex(idx, method="ffill"),
                                    "gdp_growth": gdp_growth.reindex(idx, method="ffill"),
                                    "repo_rate": repo_series.reindex(idx, method="ffill")})
print("data ready for", len(TICKERS), "tickers")

### 2. Full-history panel + walk-forward split + LightGBM

In [ ]:
FEATURES = TECH_COLS + FUND_COLS + MACRO_COLS
panels = {}
for tk in TICKERS:
    df = price_data[tk][["Close","fwd_ret"] + TECH_COLS].join(fund_data[tk]).join(macro_data[tk])
    df[FUND_COLS] = df[FUND_COLS].fillna(0.0)
    df["ticker"] = tk
    panels[tk] = df.dropna(subset=TECH_COLS + MACRO_COLS + ["fwd_ret"])

full = pd.concat(panels.values()).sort_index()
train_mask = full.index < TRAIN_END
val_mask   = (full.index >= TRAIN_END) & (full.index < VAL_END)
test_mask  = full.index >= VAL_END
print("rows:", len(full), " train/val/test:", train_mask.sum(), val_mask.sum(), test_mask.sum())

model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=31,
                           min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
                           random_state=SEED, verbose=-1)
model.fit(full.loc[train_mask, FEATURES], full.loc[train_mask, "fwd_ret"],
          eval_set=[(full.loc[val_mask, FEATURES], full.loc[val_mask, "fwd_ret"])],
          callbacks=[lgb.early_stopping(30, verbose=False)])

pred_test = model.predict(full.loc[test_mask, FEATURES])
actual_test = full.loc[test_mask, "fwd_ret"].values
mae = mean_absolute_error(actual_test, pred_test); r2 = r2_score(actual_test, pred_test)
dir_acc = (np.sign(pred_test) == np.sign(actual_test)).mean()
base_rate = (actual_test > 0).mean()
print(f"test MAE {mae:.5f}  R2 {r2:+.4f}  dir_acc {dir_acc:.3%}  (base up-rate {base_rate:.3%}, n={test_mask.sum()})")

### 3. SHAP — global importance, local explanations for real dates, dependence plot

In [ ]:
explainer = shap.TreeExplainer(model)
X_test = full.loc[test_mask, FEATURES]
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test, show=True)

In [ ]:
# local explanations for two real, meaningful dates: the single largest predicted move,
# and (if available) a date closest to a real reported earnings surprise in the test window
i_biggest = int(np.argmax(np.abs(pred_test)))
print("largest predicted move:", X_test.index[i_biggest].date(), "predicted fwd ret:", pred_test[i_biggest])
shap.plots.waterfall(shap_values[i_biggest], show=True)

top_feat = X_test.columns[np.argsort(-np.abs(shap_values.values).mean(axis=0))[0]]
print("most important feature overall:", top_feat)
shap.dependence_plot(top_feat, shap_values.values, X_test, show=True)

### 4. Sentiment-specific diagnostics on the recent, real-news-covered window

Same disclosed constraint as Notebooks 1-2: free news APIs only cover a recent window per ticker. We fetch it
fresh here, fit a small second LightGBM model with the sentiment gate as a feature, and use *that* model for
noise-robustness and temporal-dynamics analysis (the full-history model above never saw sentiment, since it
would be neutral/imputed for ~95% of its history — that would not be an honest test).

In [ ]:
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")
POLARITY_SIGN = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}
MATERIALITY_KEYWORDS = ["earnings","profit","loss","guidance","acquisition","merger","stake","ipo","dividend",
                         "buyback","lawsuit","regulatory","rbi","repo","downgrade","upgrade","rating","fraud",
                         "default","ceo","resign","results","revenue","contract","deal"]

def materiality_score(title):
    t = title.lower()
    return min(1.0, sum(1 for kw in MATERIALITY_KEYWORDS if kw in t) / 3.0)

def add_novelty(df, window=5):
    if len(df) < 2:
        df["novelty"] = 1.0; return df
    tfidf = TfidfVectorizer(stop_words="english").fit_transform(df["title"])
    novelty = []
    for i in range(len(df)):
        lo = max(0, i - window)
        novelty.append(1.0 if i == lo else float(1.0 - cosine_similarity(tfidf[i], tfidf[lo:i]).max()))
    df["novelty"] = novelty
    return df

def fetch_news_sentiment(ticker):
    try:
        news = yf.Ticker(ticker).news
    except Exception:
        news = []
    rows = [{"date": pd.to_datetime(n["content"]["pubDate"]).tz_localize(None), "title": n["content"]["title"]}
            for n in news if n.get("content", {}).get("title") and n.get("content", {}).get("pubDate")]
    if not rows:
        return pd.DataFrame(columns=["date","title","gate"])
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    sc = finbert(df["title"].tolist())
    df["polarity"] = [POLARITY_SIGN[s["label"]] * s["score"] for s in sc]
    df["materiality"] = df["title"].apply(materiality_score)
    df = add_novelty(df)
    df["gate"] = df["polarity"] * df["novelty"] * df["materiality"]
    return df

news_frames = {tk: fetch_news_sentiment(tk) for tk in TICKERS}
daily_gate = {tk: (df.groupby(df["date"].dt.floor("D"))["gate"].mean().to_frame() if not df.empty
                   else pd.DataFrame(columns=["gate"])) for tk, df in news_frames.items()}

rows = []
for tk in TICKERS:
    if daily_gate[tk].empty:
        continue
    m = panels[tk][TECH_COLS + ["fwd_ret"]].join(daily_gate[tk], how="inner").dropna()
    if len(m):
        rows.append(m.assign(ticker=tk))
sent_panel = pd.concat(rows).sort_index() if rows else pd.DataFrame()
print("sentiment-covered rows:", len(sent_panel))

In [ ]:
if len(sent_panel) >= 40:
    SENT_FEATURES = TECH_COLS + ["gate"]
    n = len(sent_panel)
    cut = int(n * 0.7)
    Xs, ys = sent_panel[SENT_FEATURES], sent_panel["fwd_ret"]
    Xs_tr, Xs_te = Xs.iloc[:cut], Xs.iloc[cut:]
    ys_tr, ys_te = ys.iloc[:cut], ys.iloc[cut:]

    sent_model = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, num_leaves=15,
                                    min_child_samples=5, random_state=SEED, verbose=-1)
    sent_model.fit(Xs_tr, ys_tr)
    base_pred = sent_model.predict(Xs_te)
    base_mae = mean_absolute_error(ys_te, base_pred)
    print(f"sentiment-window model: n_train={cut} n_test={n-cut}  test MAE {base_mae:.5f}")

    # --- noise robustness: corrupt only the 'gate' feature at inference ---
    sigmas = [0.0, 0.05, 0.1, 0.2, 0.4, 0.8]
    maes = []
    rng = np.random.default_rng(SEED)
    gate_std = Xs_te["gate"].std() or 1.0
    for s in sigmas:
        Xn = Xs_te.copy()
        Xn["gate"] = Xn["gate"] + rng.normal(0, s * gate_std, size=len(Xn))
        p = sent_model.predict(Xn)
        maes.append(mean_absolute_error(ys_te, p))
    plt.figure(figsize=(6,4))
    plt.plot(sigmas, maes, marker="o")
    plt.xlabel("sentiment-noise sigma (x feature std)"); plt.ylabel("test MAE")
    plt.title("Robustness: forecast error vs. injected sentiment noise")
    plt.tight_layout(); plt.show()

    # --- SHAP importance of the gate feature specifically ---
    sent_explainer = shap.TreeExplainer(sent_model)
    sv = sent_explainer(Xs_te)
    gate_idx = list(Xs_te.columns).index("gate")
    gate_importance = np.abs(sv.values[:, gate_idx]).mean()
    total_importance = np.abs(sv.values).mean(axis=0).sum()
    print(f"sentiment gate's share of total mean |SHAP|: {gate_importance/total_importance:.1%}")

    # --- temporal dynamics: Granger causality + cross-correlation, gate -> fwd_ret ---
    pooled = sent_panel.groupby(level=0)[["gate","fwd_ret"]].mean().dropna()
    if len(pooled) >= 15:
        try:
            gc = grangercausalitytests(pooled[["fwd_ret","gate"]], maxlag=5, verbose=False)
            print("Granger causality p-values (gate -> fwd_ret), lag: p-value")
            for lag, res in gc.items():
                print(f"  lag {lag}: p={res[0]['ssr_ftest'][1]:.4f}")
        except Exception as e:
            print("Granger test could not run (likely too few pooled time points):", e)

        ccf = [pooled["gate"].shift(lag).corr(pooled["fwd_ret"]) for lag in range(0, 6)]
        plt.figure(figsize=(6,4))
        plt.bar(range(0,6), ccf)
        plt.xlabel("lag (days, gate leads fwd_ret by this many days)"); plt.ylabel("correlation")
        plt.title("Cross-correlation: sentiment gate vs forward return")
        plt.tight_layout(); plt.show()
        print("Read p-values and correlations at face value: with n=", len(pooled),
              "pooled time points this is not a well-powered test. A null result here is a real, honest finding,"
              " not a failure of the notebook.")
    else:
        print("Too few pooled daily observations for a meaningful Granger/cross-correlation test this run.")
else:
    print(f"Only {len(sent_panel)} sentiment-covered rows this run -- skipping the noise/temporal-dynamics "
          "diagnostics rather than fitting an unreliable model on too little data.")

### 5. Illustrative decision-quality experiment: trade only when SHAP drivers agree

Algorithmic proxy for the SHAP-portfolio-rebalancing paper's human-subject finding: instead of asking a human
to trust the model more when explanations are clear, we **automate** the same idea — only act on a prediction
when its top-2 SHAP feature contributions **agree in sign** (a coherent explanation), and compare a simple
long/short daily strategy with vs without this filter, on the untouched full-history test set from Section 2.

In [ ]:
sv_test = shap_values.values  # (n, n_features), full-history model, Section 2/3
top2_sign_agree = []
for row in sv_test:
    order = np.argsort(-np.abs(row))[:2]
    top2_sign_agree.append(np.sign(row[order[0]]) == np.sign(row[order[1]]))
top2_sign_agree = np.array(top2_sign_agree)

def strategy_stats(mask):
    r = actual_test[mask] * np.sign(pred_test[mask])   # realized return of "trade in predicted direction"
    if len(r) < 2:
        return np.nan, np.nan, 0
    sharpe = (r.mean() / (r.std() + 1e-9)) * np.sqrt(252 / HORIZON)
    hit = (r > 0).mean()
    return sharpe, hit, len(r)

sh_all, hit_all, n_all = strategy_stats(np.ones(len(actual_test), dtype=bool))
sh_f, hit_f, n_f = strategy_stats(top2_sign_agree)
print(f"trade every prediction:        Sharpe {sh_all:.2f}  hit-rate {hit_all:.3%}  n={n_all}")
print(f"trade only when SHAP agrees:   Sharpe {sh_f:.2f}  hit-rate {hit_f:.3%}  n={n_f} "
      f"({n_f/n_all:.1%} of days kept)")
print("Report whichever is actually better here -- this is a real backtest on real returns, not a demonstration "
      "rigged to favor the SHAP filter.")

## Summary for a presentation slide

- **Data**: real technical/fundamental/macro history (2015-today, large sample) for the main model + real,
  current news headlines and FinBERT sentiment for the sentiment-specific diagnostics (small, disclosed sample).
- **Explainability**: SHAP TreeExplainer on a real LightGBM forecaster — global summary, local waterfalls tied
  to real dates, dependence plot, all computed directly from this run.
- **Sentiment noise robustness & temporal dynamics**: reported honestly with sample sizes; a flat noise curve
  or a non-significant Granger test is a legitimate finding to present, not something to omit.
- **Decision-quality proxy**: SHAP-agreement filter vs trade-everything, real backtest Sharpe/hit-rate from
  Section 5 — quote whichever number the run actually produced.